# Causal Language Modeling

This is a guide from the HF platform where I tried to finetune distilBERT on the ELi5 dataset.

The finetuning ran for a good 7 and a half hours but it got interupted at epoch 2.06/3. The model didnt work after I tried evaulating it after refreshing the page. Therefore, there are a few KeyboardInterupt errors that can be ignored.

This can be a great structure that anyone can refer to how to use Causal Language Modeling.

In [ ]:
pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00


## Login

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Preprocess the data

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")
eli5 = eli5.train_test_split(test_size=0.2)

README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 98.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.92MB            

data/validation1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/91772 [00:00<?, ? examples/s]

Generating validation1 split:   0%|          | 0/5446 [00:00<?, ? examples/s]

Generating validation2 split:   0%|          | 0/2375 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5411 [00:00<?, ? examples/s]

In [ ]:
eli5['train'][0]

{'q_id': '5my6k3',
 'title': 'How far are we looking?',
 'selftext': 'Let\'s say you are standing at the edge of the beach and looking outwards into the horizon. How far is this "horizon" and what are we looking at? I understand that there is a curvature to the earth, so does that mean that I\'m looking into the sky?',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dc77276', 'dc7awel'],
  'text': ["For an average height man standing on the ground with no hills etc. in the way, the horizon is about 3 miles away. So yes, past that you're looking on a low slope up into the sky even further away.",
   'The horizon is the edge of the part of the earth that is visible to you due to your height. Below it you see the Earth, above it you see the sky. This article has a very detailed explanation with calculations of the distance of the horizon, but most importantly a nice big diagram that should make it easy to understand: URL_0'],
  'score': [6, 4],
  'text_urls

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert/distilgpt2')

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
eli5 = eli5.flatten()
eli5['train'][0]

{'q_id': '5my6k3',
 'title': 'How far are we looking?',
 'selftext': 'Let\'s say you are standing at the edge of the beach and looking outwards into the horizon. How far is this "horizon" and what are we looking at? I understand that there is a curvature to the earth, so does that mean that I\'m looking into the sky?',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dc77276', 'dc7awel'],
 'answers.text': ["For an average height man standing on the ground with no hills etc. in the way, the horizon is about 3 miles away. So yes, past that you're looking on a low slope up into the sky even further away.",
  'The horizon is the edge of the part of the earth that is visible to you due to your height. Below it you see the Earth, above it you see the sky. This article has a very detailed explanation with calculations of the distance of the horizon, but most importantly a nice big diagram that should make it easy to understand: URL_0'],
 'answers.score': [6, 4],
 'a

In [ ]:
def preprocess_function(examples):
  return tokenizer([" ".join(x) for x in examples['answers.text']])

In [ ]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5['train'].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1109 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1069 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3580 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2026 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1312 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1049 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1667 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1446 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
block_size = 128

def group_texts(examples):
  concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
  total_length = len(concatenated_examples[list(examples.keys())[0]])

  if total_length >= block_size:
    total_length = (total_length // block_size) * block_size

  result = {
      k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
      for k, t in concatenated_examples.items()
  }

  result['labels'] = result['input_ids'].copy()
  return result

In [ ]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='my_awesome_eli5_clm-model',
    eval_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset['train'],
    eval_dataset=lm_dataset['test'],
    data_collator=data_collator,
    processing_class=tokenizer
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Evaulate and infer

In [ ]:
import math

eval_results = trainer.evaluate()
print(f'PerplexityL {math.exp(eval_results['eval_loss']):.2f}')

NameError: name 'trainer' is not defined

In [ ]:
trainer.push_to_hub()

In [ ]:
prompt = 'Somatic hypermutation allows the immune system to'

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="official-ak/my_awesome_eli5_clm-model")
generator(prompt)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoToeknizer.from_pretrained('official-ak/my_awesome_eli5_clm-model')
inputs - tokenizer(prompt, return_tensors='pt').input_ids

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("username/my_awesome_eli5_clm-model")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

In [ ]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)